In [1]:
from edgar import *
from edgar.documents import HTMLParser, ParserConfig, Document
from os import *
from pathlib import Path
from rich.console import Console

console = Console()

set_identity(environ.get("SEC_USER_AGENT"))
ticker = "GEHC"
# utils/ 目录的上一级是项目根目录
project_root = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd().parent
storage_path = project_root / f"workspace/portfolio/{ticker}/filings"
gehc_2022 = storage_path / "fil_0001932393-23-000025"

# gehc_2022_10K = gehc_2022 / "gehc-20221231.htm"
# parser = HTMLParser(ParserConfig(form="10-K"))
# doc : Document = parser.parse_file(gehc_2022_10K)

# print(f"sections: {len(doc.sections)}")



/opt/homebrew/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from edgar.xbrl import XBRL

# 直接从本地 filing 目录发现 XBRL 相关文件。
instance_file = next(iter(sorted(gehc_2022.glob("*_htm.xml"))), None)
if instance_file is None:
    instance_file = next(iter(sorted(gehc_2022.glob("*_ins.xml"))), None)
schema_file = next(iter(sorted(gehc_2022.glob("*.xsd"))), None)
presentation_file = next(iter(sorted(gehc_2022.glob("*_pre.xml"))), None)
calculation_file = next(iter(sorted(gehc_2022.glob("*_cal.xml"))), None)
definition_file = next(iter(sorted(gehc_2022.glob("*_def.xml"))), None)
label_file = next(iter(sorted(gehc_2022.glob("*_lab.xml"))), None)

print("instance_file:", instance_file)
print("schema_file:", schema_file)
print("presentation_file:", presentation_file)
print("calculation_file:", calculation_file)
print("definition_file:", definition_file)
print("label_file:", label_file)

if instance_file is None or schema_file is None:
    raise RuntimeError("缺少 instance/schema，无法继续加载 XBRL")

# 用 edgartools 直接构建 XBRL 对象。
xbrl = XBRL.from_files(
    instance_file=instance_file,
    schema_file=schema_file,
    presentation_file=presentation_file,
    calculation_file=calculation_file,
    definition_file=definition_file,
    label_file=label_file,
)

print("fiscal_year:", getattr(xbrl, "fiscal_year", None))
print("fiscal_period:", getattr(xbrl, "fiscal_period", None))
print("entity_info:", getattr(xbrl, "entity_info", None))

# 检查 statement 路径是否能解析出主表。
statement_methods = {
    "income_statement": getattr(xbrl.statements, "income_statement", None),
    "balance_sheet": getattr(xbrl.statements, "balance_sheet", None),
    "cash_flow_statement": getattr(xbrl.statements, "cash_flow_statement", None),
    "statement_of_equity": getattr(xbrl.statements, "statement_of_equity", None),
    "comprehensive_income": getattr(xbrl.statements, "comprehensive_income", None),
}

for name, method in statement_methods.items():
    print(f"\\n[{name}]")
    if not callable(method):
        print("method not available")
        continue
    try:
        statement = method()
        print("statement object:", statement)
        if statement is None:
            print("resolved to None")
            continue
        if hasattr(statement, "get_dataframe"):
            df = statement.get_dataframe()
            print("dataframe shape:", df.shape)
            display(df.head())
    except Exception as exc:
        print(type(exc).__name__, exc)

# 再检查 query 路径，确认 facts 本身是否存在。
for concept in ["Assets", "Revenues", "NetIncomeLoss"]:
    rows = xbrl.query().by_concept(concept).execute()
    print(f"\\nquery concept={concept}: count={len(rows)}")
    if rows:
        display(rows[:3])


Failed to resolve IncomeStatement for GE HEALTHCARE TECHNOLOGIES INC. (CIK: Unknown, Period: 2022-12-31). No matching statements found. No statements available. No statements available in XBRL data
Failed to resolve BalanceSheet for GE HEALTHCARE TECHNOLOGIES INC. (CIK: Unknown, Period: 2022-12-31). No matching statements found. No statements available. No statements available in XBRL data
Failed to resolve StatementOfEquity for GE HEALTHCARE TECHNOLOGIES INC. (CIK: Unknown, Period: 2022-12-31). No matching statements found. No statements available. No statements available in XBRL data
Failed to resolve ComprehensiveIncome for GE HEALTHCARE TECHNOLOGIES INC. (CIK: Unknown, Period: 2022-12-31). No matching statements found. No statements available. No statements available in XBRL data


instance_file: workspace/portfolio/GEHC/filings/fil_0001932393-23-000025/gehc-20221231_htm.xml
schema_file: workspace/portfolio/GEHC/filings/fil_0001932393-23-000025/gehc-20221231.xsd
presentation_file: workspace/portfolio/GEHC/filings/fil_0001932393-23-000025/gehc-20221231_pre.xml
calculation_file: None
definition_file: None
label_file: workspace/portfolio/GEHC/filings/fil_0001932393-23-000025/gehc-20221231_lab.xml
fiscal_year: None
fiscal_period: None
entity_info: {'entity_name': 'GE HEALTHCARE TECHNOLOGIES INC.', 'ticker': 'GEHC', 'identifier': '1932393', 'document_type': '10-K', 'reporting_end_date': datetime.date(2023, 1, 31), 'document_period_end_date': '2022-12-31', 'fiscal_year': '2022', 'fiscal_period': 'FY', 'fiscal_year_end_month': 12, 'fiscal_year_end_day': 31, 'annual_report': True, 'quarterly_report': False, 'amendment': False}
\n[income_statement]
statement object: None
resolved to None
\n[balance_sheet]
statement object: None
resolved to None
\n[cash_flow_statement]
met